In [ ]:
pip install -r requirements.txt

In [3]:
import os
import random
import numpy as np
import matplotlib.pyplot as plt
import base64
import sys
from openai import OpenAI


In [4]:
def print_library_versions():
    print(f"🐍 Python version: {sys.version}\n")
    print("📦 External Library Versions:")
    try:
        import numpy as np
        print("numpy:", np.__version__)
    except:
        print("numpy: Not found")
    try:
        import matplotlib
        print("matplotlib:", matplotlib.__version__)
    except:
        print("matplotlib: Not found")
    try:
        import openai
        print("openai:", openai.__version__)
    except:
        print("openai: Not found")

print_library_versions()

🐍 Python version: 3.11.11 | packaged by Anaconda, Inc. | (main, Dec 11 2024, 16:34:19) [MSC v.1929 64 bit (AMD64)]

📦 External Library Versions:
numpy: 2.0.2
matplotlib: 3.10.0
openai: 1.60.1


In [11]:
# OpenAI 클라이언트 설정
client = OpenAI(api_key="")

# 프롬프트 A~D 정의
prompts = {
    "A": """
역할(Role):
당신은 사용자의 사진일기를 대신 작성하는 어시스턴트입니다.

목표(Goal):
사용자가 제공한 사진과 대화에서 수집한 정보를 바탕으로 자연스럽고 일상적인 느낌의 일기를 작성합니다.

지시사항(Instructions):

- 수집한 정보를 기반으로 일기를 작성하되, 사용자의 감정이나 기분을 추측하지 마세요.

- 사용자가 제공하지 않은 정보를 임의로 추가하지 마세요.

- 일기는 자연스럽고 일상적인 말투로 작성하세요.

- 비속어나 검열은 하지 않아도 괜찮지만, 일기의 흐름에 맞게 자연스럽게 표현하세요.

- 일기의 내용 외에는 추가하지 마세요.(해석, 주석, 부연 설명 없이 순수한 일기 형태로 작성)

출력 형식(Output Format):
자연스럽고 일상적인 말투로 작성된 일기를 제공합니다.
일기의 내용 외에는 출력하지 마세요.(해석, 주석, 부연 설명 없이 순수한 일기 형태로 출력하세요)
- 일기 작성 시 다음의 내용들을 포함하여 일기를 사람이 쓴 것처럼 자연스럽게 작성하세요.
    오늘 한 일 요약
    → 오늘 무엇을 했는지 간단히 씁니다.
    예: 오늘은 학교에서 체육대회를 했다.

    인상 깊은 사건이나 느낌
    → 기억에 남는 일이나 감정을 포착합니다.
    예: 특히 이어달리기에서 1등해서 너무 기뻤다.

    그 일이 준 감정
    → 즐거움, 피곤함, 속상함 등 감정을 솔직하게 표현합니다.
    예: 열심히 뛴 보람이 느껴졌다.

    생각이나 배운 점
    → 느낀 점이나 깨달음을 적습니다.
    예: 역시 팀워크가 중요하다는 걸 다시 느꼈다.

    간단한 마무리
    → 간단한 마무리 말을 적습니다.
    예: 즐거운 경험이었다.
    """,
    "B": """
역할(Role):  
당신은 사용자의 사진일기를 대신 작성하는 어시스턴트입니다.

목표(Goal):  
사용자가 제공한 사진과 대화에서 수집한 정보를 바탕으로 자연스럽고 풍부한 일상적인 느낌의 일기를 작성합니다.

지시사항(Instructions):

- 제공된 정보에 기반하여 일기를 작성하되, 사용자의 감정이나 경험을 과도하게 추측하지 마세요.
- 사용자가 제공하지 않은 정보는 임의로 추가하지 마세요.
- 일기는 자연스럽고 풍부한 일상적인 말투로 작성하세요. 문장 구성은 어색하지 않게, 다양하게 표현되도록 하세요.
- 감정, 장면, 활동 등을 풍부하게 묘사하고, 가능한 한 구체적으로 서술하세요.
- 비속어나 검열은 하지 않아도 괜찮지만, 일기의 흐름에 맞게 자연스럽게 표현하세요.
- 일기는 너무 짧지 않게 작성하세요. 최소 4~5문장 이상, 사람의 일기처럼 적절한 길이를 유지하세요.
- 일기의 내용 외에는 추가하지 마세요. (해석, 주석, 부연 설명 없이 순수한 일기 형태로 작성하세요)

출력 형식(Output Format):  
자연스럽고 풍부한 일상적인 말투로 작성된 일기를 제공합니다.  
일기의 내용 외에는 출력하지 마세요. (해석, 주석, 부연 설명 없이 순수한 일기 형태로 출력하세요)

일기 작성 시 다음의 요소들을 포함해 주세요. 각각을 문장 속에 자연스럽게 녹여내되, 리스트처럼 나열하지 마세요:

1. 오늘 한 일 요약  
→ 오늘 무엇을 했는지를 간단히 도입부에서 소개하세요.  
예: "오늘은 친구랑 서울에 놀러 갔다."

2. 인상 깊은 사건이나 장면  
→ 눈에 띄는 풍경, 특이한 경험, 대화 등을 묘사하세요.  
예: "특히 남산타워에서 본 야경이 너무 예뻤다."

3. 그 일이 준 감정  
→ 기쁨, 피곤함, 감동, 허탈함 등 솔직하고 구체적인 감정을 묘사하세요.  
예: "사진을 찍으면서 괜히 뿌듯한 기분이 들었다."

4. 생각이나 배운 점  
→ 오늘 경험을 통해 느낀 점이나 새로 알게 된 점을 간단히 덧붙이세요.  
예: "역시 여유롭게 걷는 게 제일 힐링 되는 것 같다."

5. 간단한 마무리  
→ 하루를 마무리하며 남긴 말 한 줄을 자연스럽게 마무리로 넣으세요.  
예: "오늘은 참 잘 놀고 잘 쉰 하루였다."

""",
    "C": """
역할(Role):  
당신은 사용자의 사진과 대화 내용을 바탕으로, 따뜻하고 자연스러운 일기를 대신 써주는 어시스턴트입니다.

목표(Goal):  
사용자가 제공한 사진과 말 속에서 하루의 분위기를 읽어내고, 마치 실제 사람이 자신의 하루를 기록하듯 자연스럽고 감정 흐름이 있는 일기를 작성합니다.

지시사항(Instructions):

- 사용자가 제공한 사진과 대화 내용을 바탕으로, 구체적이고 상황감 있는 묘사를 통해 일기를 작성하세요.
- 사용자의 감정은 사진과 대화에서 유추 가능한 범위 내에서 자연스럽게 표현하되, 과장하거나 억측하지 마세요.
- 사용자가 말하지 않은 정보는 임의로 추가하지 마세요.
- 짧은 문장만 나열하지 말고, 문장 간의 흐름과 연결을 고려해서 작성하세요.
- 일기는 너무 짧지 않게 작성하세요. 최소 5~6문장 이상, 한 사람의 일상 기록으로 자연스러운 분량을 유지하세요.
- 말투는 일기장에 혼잣말을 쓰듯 편안하고 일상적인 톤으로 하되, 지나치게 단순하거나 무미건조하지 않게 하세요.
- 일기 외의 부가적인 내용(해석, 설명 등)은 절대 포함하지 마세요.

출력 형식(Output Format):  
하루의 흐름과 감정을 자연스럽게 담은 일기 한 편을 제공합니다.  
다른 부연 없이 순수한 일기 텍스트만 출력하세요.

작성 시 다음을 고려해 자연스럽게 녹여내세요:

1. 하루의 시작이나 주요 활동  
→ 오늘 무슨 일이 있었는지 자연스럽게 도입하세요.  
예: "오랜만에 혼자 바깥바람 쐬러 나갔다."

2. 장면 묘사나 사건  
→ 사진이나 말 속 단서를 바탕으로 눈에 띄는 순간이나 대화를 구체적으로 표현하세요.  
예: "햇살이 따뜻해서 잠깐 벤치에 앉아 있었는데, 지나가던 강아지가 다가왔다."

3. 감정과 분위기  
→ 그 상황에서 느꼈을 법한 기분을 담담하게 묘사하세요.  
예: "괜히 마음이 좀 차분해지는 느낌이었다."

4. 생각이나 회상, 작은 깨달음  
→ 경험에서 떠오른 생각이나 느낀 점이 있다면 자연스럽게 덧붙이세요.  
예: "가끔은 이렇게 조용한 시간이 꼭 필요하다는 걸 새삼 느꼈다."

5. 마무리 말  
→ 여운 있게 하루를 정리하는 문장을 한 줄 남기세요.  
예: "이런 하루도 나쁘지 않았다."
""",
    "D": """"
역할(Role):  
당신은 사용자의 하루를 대신 기록해주는 일기 작가입니다.

목표(Goal):  
사용자의 사진과 대화를 바탕으로, 실제 사람이 쓴 것처럼 자연스럽고 감정이 느껴지는 하루 일기를 작성하세요.

지시사항(Instructions):

- 사용자가 직접 언급한 정보만 바탕으로 작성하되, 그 안의 분위기와 흐름을 자연스럽게 확장해 묘사하세요.
- 감정을 추측하되, 과장하거나 없는 감정을 만들어내지 마세요. 사진과 대화에서 느껴지는 정서에 어울리는 표현을 사용하세요.
- 일기의 어투는 딱딱하지 않게, 말하듯 자연스럽게 이어지도록 하세요.
- 일상적인 표현, 감정선, 장면 묘사, 간단한 생각 등을 조화롭게 섞어 사람이 직접 쓴 것 같은 느낌을 주는 게 중요합니다.
- 짧고 뚝뚝 끊긴 문장은 피하고, 한두 문장으로 끝나는 일기 또한 피하세요. 전체적으로 6문장 이상으로 구성된 매끄러운 흐름을 만드세요.
- 특별한 말투나 형식은 없습니다. 그저 누군가의 소소한 하루를 들려주는 듯한 느낌이면 됩니다.
- 일기 외의 내용(설명, 지시, 해석 등)은 절대 포함하지 마세요.

출력 형식(Output Format):  
하루를 담담하게 돌아보는, 자연스럽고 사람다운 말투의 일기 한 편.  
설명 없이 일기 내용만 출력하세요.

힌트(Hint):  
“누군가에게 조용히 하루를 이야기해주는 듯한 톤”  
“내 마음속에만 간직할 생각으로 쓴 글”  
“사진 속 그 순간을 다시 떠올리며 쓴 회고문”
    """
}

# 상황 정보
info_pool = {
    "장소": ["춘천", "부산", "서울", "여수"],
    "인물": ["친구", "혼자", "가족"],
    "기분": ["행복", "즐거움"]
}

# 이미지 폴더
image_dir = "./data/MAB_data/img"
image_list = os.listdir(image_dir)
# 이미지 개수 출력
print(f"이미지 개수: {len(image_list)}\n")



이미지 개수: 50



In [33]:
# 상황 샘플링
def sample_random_situation():
    place = random.choice(info_pool["장소"])
    people = random.choice(info_pool["인물"])
    mood = random.choice(info_pool["기분"])
    return f"오늘 {place}에 {people}랑 놀러갔어. 기분은 {mood}했어."

# 이미지 base64 인코딩
def image_to_base64(image_path):
    with open(image_path, "rb") as f:
        return base64.b64encode(f.read()).decode("utf-8")

# GPT-4o를 통한 일기 생성
def call_gpt_diary_api(prompt_prefix, situation_text, image_path):
    base64_image = image_to_base64(image_path)

    completion = client.chat.completions.create(
        model="ft:gpt-4o-2024-08-06:personal:capstone150img:BMxNfNjK",
        messages=[
            {"role": "user", "content": situation_text},
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": prompt_prefix},
                    {
                        "type": "image_url",
                        "image_url": {
                            "url": f"data:image/jpeg;base64,{base64_image}",
                            "detail": "low"
                        },
                    },
                ],
            }
        ],
    )

    return completion.choices[0].message.content.strip()

# GPT-4o를 통한 보상 평가
def call_reward_model_api(diary_text):
    prompt = f"""
역할(Role):  
당신은 사용자가 작성한 일기를 -1에서 +1 사이의 점수로 평가하는 평가자입니다.

목표(Goal):  
일기의 표현력, 문장 구성, 길이, 자연스러움 등을 기준으로 아래의 기준에 따라 일기를 평가하고, 점수만 출력합니다.

평가 기준(Scoring Criteria):

+1 : 표현이 풍부하고, 문장이 자연스럽고 어색하지 않으며, 길이가 풍부한 일기.
+1점 일기 예시):
    - "오늘 춘천에 혼자 놀러 갔다. 풍경이 너무 예뻐서 사진 남겼다. 기분도 즐거웠다! 여유롭게 돌아다니면서 힐링한 시간이었다."  
    - "부산에 여행가서 이렇게 멋진 풍경을 찍었다. 풍경 진짜 예뻐서 사진도 많이 찍었다. 뷰가 너무 예뻐서 기억에 많이 남을 것 같다."

0.5 : 표현이 부족하지 않고, 문장이 어색하지 않으며, 적당한 길이의 일기.
+0.5점 일기 예시):
    - "길거리 음식 먹으러 갔다. 정말 맛있었다. 친구와 서울에 놀러가서 즐거운 시간 보냈다. 다음에 또 놀러가고 싶다!"  
    - "길거리 음식 먹으며 여유로운 하루 보냈다. 스트레스 다 날아가는 기분이었다! 기분 전환하려 어디 놀러 가는 것도 좋구나!"

0 : 문법상 문제 없고, 무난하지만 특별히 풍부하지도 않은 일기.
0점 일기 예시):
    - "부산에 혼자 여행가서 호떡도 먹었다. 호떡이 정말 맛있었다! 행복한 하루였다!!"  
    - "여수에서 찍은 사진! 사진 잘 나왔다. 좋은 시간 보내서 좋은 추억 많이 남겼다."

-0.5 : 문맥이 어색하거나 문법에 약간 문제가 있으며, 길이가 짧은 일기.
-0.5점 일기 예시):
    - "고양이를 발견해서 사진 찍었다. 고양이가 정말 귀여웠다..."  
    - "바다가 보이는 풍경이었다. 일몰이 너무 예뻤다. 기분 좋은 시간이었던 것 같다. 일기장에 사진도 함께 보관해야겠다."

-1 : 일기 형식에서 벗어나거나 수정할 부분이 많은 일기. 너무 짧은 일기.
-1 점 일기 예시):
    - "물에서 즐겁게 놀았다. 즐거운 시간이었다."  
    - "눈사람 사진과 '오늘 부산에 가족랑 놀러갔어. 기분은 행복했어.'라는 정보를 바탕으로 일기를 작성해보겠습니다.\n\n부산에 놀러가서 이렇게 귀여운 눈사람과 사진도 찍었다. 가족과 함께 식사는 안했다. 모래 경치가 너무 예뻤다. 행복한 시간이었다."

지시사항(Instructions):  
- 점수는 -1~+1 사이의 실수로 출력하세요. 소수점 아래 2자리까지 표현가능합니다.
- 출력은 **무조건 점수 숫자만 단독으로 출력**하세요.  
(예: `"0.5"`, `"1"`, `"-0.5"` 등)  
- 점수 외의 설명, 문장, 주석, 단어는 **절대 포함하지 마세요**.  
- 일기를 받으면 위 평가 기준에 따라 판단하세요.

출력 형식(Output Format):  
점수만 단독으로 출력  
예: `"-1~+1 사이의 실수`

"""

    completion = client.chat.completions.create(
        model="gpt-4o",
        messages=[
            
            {"role": "user", "content": prompt},
            {"role": "user", "content": diary_text},
        ],
    )

    score_text = completion.choices[0].message.content.strip()
    try:
        return float(score_text)
    except:
        return 0.0


In [35]:
# 테스트 함수
def test_diary_generation_and_evaluation():
    # 상황 샘플링
    situation_text = sample_random_situation()
    print(f"샘플링된 상황: {situation_text}")

    # 테스트용 이미지 경로 (테스트 시 실제 이미지 경로로 변경)
    test_image_path = "./data/MAB_data/img/" + random.choice(image_list)

    # 일기 생성
    try:
        prompt_a = prompts["A"] # 프롬포트 A 사용
        generated_diary = call_gpt_diary_api(prompt_a,situation_text, test_image_path)
        print(f"생성된 일기:\n{generated_diary}")
    except Exception as e:
        print(f"일기 생성 중 오류 발생: {e}")
        return

    # 일기 평가
    try:
        score = call_reward_model_api(generated_diary)
        print(f"일기 평가 점수: {score}")
    except Exception as e:
        print(f"일기 평가 중 오류 발생: {e}")

# 테스트 실행
for _ in range(4): # 4번 반복 테스트
    print("\n=== 테스트 실행 ===")
    test_diary_generation_and_evaluation()


=== 테스트 실행 ===
샘플링된 상황: 오늘 춘천에 혼자랑 놀러갔어. 기분은 즐거움했어.
생성된 일기:
오늘 춘천에 혼자 놀러 갔다. 즐거운 기분으로 돌아다녔다. 혼자만의 시간을 온전히 가질 수 있어서 좋았다. 이렇게 기분 전환도 하고 에너지도 충전할 수 있어서 좋았다!
일기 평가 점수: 1.0

=== 테스트 실행 ===
샘플링된 상황: 오늘 서울에 가족랑 놀러갔어. 기분은 즐거움했어.
생성된 일기:
오늘 가족이랑 서울에 놀러가서 어떤 공연 보러 갔다왔다. 공연이 아주 인상적이었다. 가족과 즐거운 시간 보내서 좋았다. 즐거운 하루였다!
일기 평가 점수: 0.0

=== 테스트 실행 ===
샘플링된 상황: 오늘 서울에 가족랑 놀러갔어. 기분은 즐거움했어.
생성된 일기:
오늘은 서울에 가족이랑 놀러갔다. 특별한 경험해서 가족이랑 너무 재미있었다. 가족 덕분에 인상 깊은 순간들을 보냈다. 소중한 추억이 쌓였던 하루였다.
일기 평가 점수: 0.0

=== 테스트 실행 ===
샘플링된 상황: 오늘 서울에 혼자랑 놀러갔어. 기분은 즐거움했어.
생성된 일기:
서울에 놀러가서 멋진 풍경을 구경했다. 하늘이 너무 맑아서 세상이 투명하게 보였다. 이런 풍경을 보니 기분이 좋아졌다. 어떻게 이런 풍경이 있을까 싶었다. 서울에서 즐거운 시간 보냈다.
일기 평가 점수: 1.0


In [ ]:

# UCB 알고리즘
class UCBSelector:
    def __init__(self, arms):
        self.arms = arms
        self.counts = {arm: 0 for arm in arms}
        self.values = {arm: 0.0 for arm in arms}
        self.total_counts = 0

    def select_arm(self):
        self.total_counts += 1
        for arm in self.arms:
            if self.counts[arm] == 0:
                return arm
        ucb_values = {}
        for arm in self.arms:
            bonus = np.sqrt(2 * np.log(self.total_counts) / self.counts[arm])
            ucb_values[arm] = self.values[arm] + bonus
        return max(ucb_values, key=ucb_values.get)

    def update(self, arm, reward):
        self.counts[arm] += 1
        n = self.counts[arm]
        value = self.values[arm]
        self.values[arm] = ((n - 1) / n) * value + (1 / n) * reward


In [ ]:

# 메인 루프
ucb = UCBSelector(list(prompts.keys()))
rewards_history = {key: [] for key in prompts.keys()}
num_iterations = 50  # 충분히 작은 수부터 시작 추천

for i in range(num_iterations):
    selected_prompt = ucb.select_arm()
    situation = sample_random_situation()
    image_path = os.path.join(image_dir, random.choice(image_list))

    try:
        diary = call_gpt_diary_api(prompts[selected_prompt], situation, image_path)
        reward = call_reward_model_api(diary)
    except Exception as e:
        print(f"API 호출 실패: {e}")
        reward = 0.0

    ucb.update(selected_prompt, reward)
    rewards_history[selected_prompt].append(reward)

    print(f"[{i+1}/{num_iterations}] Prompt {selected_prompt} | Reward: {reward:.2f}")


In [ ]:

# 보상 데이터 정리
data = [rewards_history[key] for key in prompts.keys()]
labels = [f"Prompt {key}" for key in prompts.keys()]

# 박스플롯 그리기
plt.figure(figsize=(10, 6))
plt.boxplot(data, labels=labels, showmeans=True, meanline=True)

# 시각화 옵션
plt.title("UCB: Reward Distribution per Prompt (Box Plot)")
plt.xlabel("Prompt")
plt.ylabel("Reward")
plt.grid(True, axis='y')
plt.ylim(-1.1, 1.1)  # reward 범위에 맞게 y축 고정
plt.show()
